<div class="alert alert-block alert-info">
<b>ℹ️ Note on Execution Order</b><br>
This notebook expects the preprocessed data to already be saved. If you haven't yet, please run <code>02-preprocessing.ipynb</code> first so the required data is available to load.
</div>

In [ ]:
!pip install -qq pyspark ray raydp

import os
import sys
import logging
from pathlib import Path

os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"

import ray
import raydp

# Python-side loggers
logging.getLogger("ray").setLevel(logging.ERROR)
logging.getLogger("ray.data").setLevel(logging.ERROR)

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    Word2Vec,
    VectorAssembler,
    StandardScaler,
    StopWordsRemover,
    Imputer,
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
# Clean notebook reruns
ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    logging_level=logging.ERROR,
    log_to_driver=False,
)

In [ ]:
spark = raydp.init_spark(
    app_name="FineWeb_Spark_on_Ray",
    num_executors=31,
    executor_cores=1,
    executor_memory="4GB",
)

In [ ]:
from fineweb_spark.paths import INTERIM_DATA_DIR

# Load preprocessed data
df = spark.read.parquet(str(INTERIM_DATA_DIR / "fineweb_preprocessed.parquet")) \
          .select("text", "token_count", "label") \
          .sample(fraction=0.2, seed=42)

In [ ]:
print("===== DATASET INFO =====")

# number of rows
row_count = df.count()
print(f"Rows: {row_count:,}")

# number of columns
col_count = len(df.columns)
print(f"Columns: {col_count}")

# schema
print("\nSchema:")
df.printSchema()

# Estimated dataset size:
sample_rows = 1000
sample_pdf = df.limit(sample_rows).toPandas()

row_size_bytes = sample_pdf.memory_usage(deep=True).sum() / sample_rows
estimated_size_gb = (row_size_bytes * row_count) / (1024**3)

print("===== DATASET SIZE =====")
print(f"Estimated dataset size: {estimated_size_gb:.2f} GB")

print("\n===== DATASET EXAMPLES =====")
df.show(10, truncate=80)

In [ ]:
# # Split into Train / Validation / Test (60 / 20 / 20)
train_data, val_data, test_data = df.randomSplit([0.6, 0.2, 0.2], seed=42)

print(f"Train: {train_data.count()}  Val: {val_data.count()}  Test: {test_data.count()}")

In [ ]:
# Split raw text into lowercase word tokens, removing punctuation
tokenizer = RegexTokenizer(inputCol="text", outputCol="raw_words", pattern="\\W+", toLowercase=True)

# Remove common English stop words to reduce noise and vocabulary size.
remover = StopWordsRemover(inputCol="raw_words", outputCol="filtered_words")

# Map filtered words to 10-dimensional dense vectors. Words appearing fewer than 500 times are ignored.
word2vec = Word2Vec(vectorSize=10, minCount=500, inputCol="filtered_words", outputCol="text_features")

# Impute missing token_count values using the mean strategy, creating a new column token_count_imputed
imputer = Imputer(inputCols=["token_count"], outputCols=["token_count_imputed"], strategy="mean")

# Assemble the imputed token_count into a vector for scaling
vec_assembler_num = VectorAssembler(inputCols=["token_count_imputed"], outputCol="token_vec")
scaler = StandardScaler(inputCol="token_vec", outputCol="token_scaled", withStd=True, withMean=True)

# Combine text features and scaled token_count into a single feature vector for modeling
final_assembler = VectorAssembler(
    inputCols=["text_features", "token_scaled"], 
    outputCol="features"
)

In [ ]:
feature_pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    word2vec,
    imputer,
    vec_assembler_num,
    scaler,
    final_assembler
])

In [ ]:
%%time
print("Fitting preprocessing pipeline...")
feature_model = feature_pipeline.fit(train_data)
print("Preprocessing pipeline fit complete.")

In [ ]:
# Transform all splits and cache prepared datasets
print("Transforming and caching datasets...")

train_prepared = feature_model.transform(train_data).cache()
val_prepared   = feature_model.transform(val_data).cache()
test_prepared  = feature_model.transform(test_data).cache()

# materialize cache
train_prepared.count()
val_prepared.count()
test_prepared.count()

print("Cached prepared split sizes:")
print(f"Train prepared: {train_prepared.count():,}")
print(f"Val prepared  : {val_prepared.count():,}")
print(f"Test prepared : {test_prepared.count():,}")

In [ ]:
# inspect columns
train_prepared.select("features", "label").show(5, truncate=85)

In [ ]:
# Define model 1
rf1 = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=5,
    maxDepth=2
)

In [ ]:
%%time
print("Training Model 1...")
model1 = rf1.fit(train_prepared)
print("Model 1 complete.")

In [ ]:
# Define model 2
rf2 = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=20,
    maxDepth=8
)

In [ ]:
%%time
print("Training Model 2...")
model2 = rf2.fit(train_prepared)
print("Model 2 complete.")

In [ ]:
# Evaluate both models
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# Model 1 predictions
train_preds1 = model1.transform(train_prepared)
val_preds1   = model1.transform(val_prepared)
test_preds1  = model1.transform(test_prepared)

train_acc1 = evaluator.evaluate(train_preds1)
val_acc1   = evaluator.evaluate(val_preds1)
test_acc1  = evaluator.evaluate(test_preds1)

print("\nModel 1 — RF (numTrees=5, maxDepth=2)")
print(f" Train Accuracy: {train_acc1:.4f}")
print(f" Val Accuracy  : {val_acc1:.4f}")
print(f" Test Accuracy : {test_acc1:.4f}")


# Model 2 predictions
train_preds2 = model2.transform(train_prepared)
val_preds2   = model2.transform(val_prepared)
test_preds2  = model2.transform(test_prepared)

train_acc2 = evaluator.evaluate(train_preds2)
val_acc2   = evaluator.evaluate(val_preds2)
test_acc2  = evaluator.evaluate(test_preds2)

print("\nModel 2 — RF (numTrees=20, maxDepth=8)")
print(f" Train Accuracy: {train_acc2:.4f}")
print(f" Val Accuracy  : {val_acc2:.4f}")
print(f" Test Accuracy : {test_acc2:.4f}")

In [ ]:
# Compare results
print("\n--- Comparison ---")
print(f"{'Model':<35} {'Train':>8} {'Val':>8} {'Test':>8}")
print(f"{'RF (numTrees=5, maxDepth=2)':<35} {train_acc1:>8.4f} {val_acc1:>8.4f} {test_acc1:>8.4f}")
print(f"{'RF (numTrees=20, maxDepth=8)':<35} {train_acc2:>8.4f} {val_acc2:>8.4f} {test_acc2:>8.4f}")

In [ ]:
# Pick best model
results = [
    ("RF (numTrees=5, maxDepth=2)", model1, test_acc1),
    ("RF (numTrees=20, maxDepth=8)", model2, test_acc2),
]

best_name, best_model, best_acc = max(results, key=lambda x: x[2])

print("\nBest model:")
print(f" Name          : {best_name}")
print(f" Test accuracy : {best_acc:.4f}")

In [ ]:
# Save model
from fineweb_spark.paths import MODELS_DIR

best_model.write().overwrite().save(str(MODELS_DIR / "best_model1"))

print(f"Model saved")

In [ ]:
# Cleanup cache and shutdown the ray
train_prepared.unpersist()
val_prepared.unpersist()
test_prepared.unpersist()

ray.shutdown()
print("\nDone.")

## Fitting Analysis

**1. Where does the model sit on the underfitting / overfitting spectrum?**

After filtering and 20% sampling the dataset contains **1,935,821** rows. The raw `int_score` distribution is only **3 classes** (3, 4, 5) — not 0–4 as originally expected. Stratified sampling (fractions defined for keys 0–4) retains only classes **3** and **4** (int_score=5 is absent from the fractions dict and therefore dropped by `sampleBy`), yielding an effective **2-class** classification problem:

| int_score | After sampling |
|---|---|
| 3 | 83,826 |
| 4 | 76,394 |

**Train / Val / Test split: 96,141 / 32,063 / 32,016**

The majority-class baseline (always predicting class 3) is ≈ **52.3%**.

Actual accuracy results:

| Model | Train | Val | Test | Train–Test Gap |
|---|---|---|---|---|
| RF (numTrees=5, maxDepth=2) | 0.6219 | 0.6197 | 0.6189 | 0.003 |
| RF (numTrees=20, maxDepth=8) | 0.6998 | 0.6870 | 0.6867 | 0.013 |

**Model 1** has a train–test gap of only **0.3 pp** — virtually no overfitting, but the shallow trees (maxDepth=2) limit capacity and leave the model closer to the underfitting end. Test accuracy of 61.89% is only ~9.6 pp above the majority-class baseline.

**Model 2** shows a wider train–test gap of **1.31 pp**, indicating mild overfitting as the deeper, more numerous trees begin to capture training-specific noise. However, the gap is still manageable and the model delivers meaningfully better accuracy.

**2. Build at least one model with different hyperparameters and compare results**

Two Random Forest models were trained:
- **Model 1**: `numTrees=5`, `maxDepth=2` — very shallow trees; almost no overfitting but limited discriminative power
- **Model 2**: `numTrees=20`, `maxDepth=8` — deeper trees with more estimators; substantially higher model capacity

Model 2 outperforms Model 1 on every split (train, val, and test), gaining roughly **+7.8 pp** in test accuracy at the cost of a mildly wider generalization gap.

**3. Which model performs best and why?**

**Model 2** achieves the best test accuracy (**0.6867** vs. 0.6189). The increased depth allows each tree to learn higher-order feature interactions between Word2Vec embeddings and scaled token count, while 20 trees provide stronger ensemble variance reduction compared to just 5. The train–test gap (1.31 pp) is acceptable for this data scale and confirms that the accuracy gain reflects genuine generalization rather than pure memorization.

**4. What are the next models planned for Milestone 4 and why?**

- **Gradient Boosted Trees (GBTClassifier)**: Sequential boosting iteratively corrects residual errors from prior trees, which should close the remaining accuracy gap for this 2-class boundary (score 3 vs. 4). We will tune `maxIter` and `stepSize`.
- **XGBoost via SparkXGBClassifier**: Native L1/L2 regularization and efficient distributed training should further address the mild overfitting seen in Model 2.
- **Feature Engineering**: Adding TF-IDF vectors, sentence count, average word length, and punctuation density alongside Word2Vec to capture stylistic signals beyond token-level semantics.
- **Restore int_score=5**: Fix the fractions dict to include key `5` so the rare top-quality class is also considered in training, and evaluate whether a 3-class model can be meaningfully trained.


## Conclusion

**1. What is the conclusion of the 1st model?**

Our first model — a Random Forest classifier with `numTrees=5` and `maxDepth=2`, trained on 10-dimensional Word2Vec text embeddings and a scaled token-count feature — achieves **61.89% test accuracy** on what turned out to be an effective **2-class** educational quality scoring task (`int_score` 3 vs. 4). The raw dataset after filtering contains only three score values (3, 4, and 5); the stratified sampling step inadvertently drops `int_score=5` because it is absent from the fractions dict, leaving a binary classification problem. The train–test gap is negligible (0.3 pp), confirming that the shallow model generalizes well but is capacity-constrained: ~62% accuracy is only about 9.6 pp above the majority-class baseline of 52.3%. The pipeline design is validated — Word2Vec embeddings plus token-count carry real signal — but the current model is too shallow to exploit it fully.

**2. What can be done to possibly improve it?**

- **Increase model capacity**: As demonstrated by Model 2 (`numTrees=20`, `maxDepth=8`), deeper and more numerous trees raise test accuracy to **68.67%** (+6.78 pp) with only mild overfitting (1.31 pp train–test gap). Further increasing depth and trees — or switching to boosted ensembles — should continue to yield gains.
- **Fix the stratified sampling**: Add `5: 1.0` (or a suitable fraction) to the fractions dict so that `int_score=5` examples are retained, enabling a proper 3-class classification that matches the actual data distribution.
- **Richer text features**: Replacing or augmenting 10-dimensional Word2Vec with higher-dimensional vectors (50–100d), TF-IDF scores, sentence count, average word length, and punctuation density would provide richer representations of educational quality.
- **More training data**: The pipeline trains on only ~20% of the corpus (~96 K examples after splits). Scaling to a larger fraction would expose the model to more linguistic diversity and reduce variance.
- **Hyperparameter tuning**: A systematic `CrossValidator` + `ParamGridBuilder` sweep over `numTrees`, `maxDepth`, `minInstancesPerNode`, and Word2Vec `vectorSize` would find the optimal capacity without manual iteration.
- **Better embeddings**: Pre-trained sentence embeddings (e.g., GloVe, FastText, or projected BERT representations) would yield substantially richer text representations than a Word2Vec model trained on a filtered 20% sample.

**3. How did distributed computing help with this task?**

The FineWeb-Edu 10BT sample is composed of 14 Parquet files totalling billions of tokens — far too large to load into a single machine's memory. Running on SDSC Expanse with Spark (31 executor cores) enabled:
- **Parallel data loading and filtering** across 31 executor cores, reducing I/O bottlenecks while processing 1.9 M rows after cleaning the 20% sample
- **Distributed Word2Vec training** using partitioned gradient updates, completing in minutes instead of hours on so many documents
- **Parallel Random Forest construction** (Model 1 and Model 2 each took ~13 min 15 s) with trees built concurrently across executors
- **Scalable inference** via `.transform()` applied partition-by-partition — evaluating three splits (train 96 K / val 32 K / test 32 K) in just ~6 seconds for Model 1
- **Checkpointing** at two key stages (after cleaning and after stratified sampling) broke long lineage chains and prevented redundant recomputation during evaluation

Even the 20% sample used here (~1.9 M rows before stratified sampling) would challenge a local machine once Word2Vec and pipeline preprocessing are factored in. Training on the full 14-file dataset without distributed computing would be entirely infeasible.
